In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import requests
import os
from google.colab import userdata
from google.colab import drive
drive.mount('/content/drive')
import json
from IPython.display import clear_output

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
tng_api_key = userdata.get('TNG_API_KEY')
baseUrl = 'http://www.tng-project.org/api/'
headers = {"api-key":tng_api_key}

In [ ]:
def get(path, params=None, out_filename=None):
    headers = {"api-key":tng_api_key}
    r = requests.get(path, params=params, headers=headers)
    r.raise_for_status()

    if out_filename is not None:
        with open(out_filename, 'wb') as f:
            f.write(r.content)
        return out_filename

    if r.headers['content-type'] == 'application/json':
        return r.json()

    if 'content-disposition' in r.headers:
        filename = r.headers['content-disposition'].split("filename=")[1]
        with open(filename, 'wb') as f:
            f.write(r.content)
        return filename

    return r

In [ ]:
r = get(baseUrl)

for simulation in r['simulations']: #only get TNG50
    if simulation['name'] == 'TNG50-1':
        url = simulation['url']
        break

tng50 = get(url)

url = 'http://www.tng-project.org/api/TNG50-1/snapshots/z=1.8/'
snapshot = get(url)

In [ ]:
drive_path = '/content/drive/MyDrive/docs'
os.makedirs(drive_path, exist_ok=True)

bhmdot = np.zeros(snapshot['num_groups_subfind'])
sfr = np.zeros(snapshot['num_groups_subfind'])

start_index = 422000
skipped_count = 0
processed_count = 0
total_subhalos = snapshot['num_groups_subfind']
log_messages = []

for i in range(start_index, total_subhalos):
    out_filename = os.path.join(drive_path, f'sub{i}.json')

    if os.path.exists(out_filename):
        skipped_count += 1
        message = f"Skipped sub{i} (already exists)" # Added message for skipped files
    else:
        sub_url = f"http://www.tng-project.org/api/TNG50-1/snapshots/{snapshot['number']}/subhalos/{i}/"
        try:
            result = get(sub_url, out_filename=out_filename)
            if result == out_filename and os.path.exists(out_filename):
                message = f"Saved sub{i}"
                processed_count += 1
            else:
                message = f"Couldn't save sub{i}"
        except requests.exceptions.RequestException as e:
                message = f"Error with sub{i}: {e}"

    log_messages.append(message)
    log_messages = log_messages[-15:] # Keep only the last 15 messages

    clear_output(wait=True)
    for msg in log_messages:
        print(msg)

KeyboardInterrupt: 